In [1]:
'''
2021/11/18
Sample solution for Lab7 COMP3055 UNNC
Environments:	Python = 3.7.4
		numpy = 1.19.1
'''

from functools import reduce

class Perceptron(object):
    """
    Simple Perceptron implementation.
    """
    def __init__(self, input_num, activator):
        """
        Initialize the Perceptron.
        :param input_num: Number of input features.
        :param activator: Activation function.
        """
        self.activator = activator
        # Initialize weights to 0.0 for each input feature
        self.weights = [0.0 for _ in range(input_num)]
        # Initialize bias to 0.0
        self.bias = 0.0

    def predict(self, input_vec):
        """
        Predict the output for a given input vector.
        :param input_vec: Input vector.
        :return: Predicted class label (0 or 1).
        """
        # Process Flow:
        # 1. Zip input vector with weights: [(x1, w1), (x2, w2), ...]
        # 2. Multiply pairs: [x1*w1, x2*w2, ...]
        # 3. Sum results and add bias: (x1*w1 + x2*w2 + ...) + bias
        # 4. Apply activation function (step function)
        
        # Calculate the weighted sum: sum(w_i * x_i) + bias
        # Then apply the activation function
        return self.activator(
            reduce(lambda a, b: a + b,
                   list(map(lambda tp: tp[0] * tp[1], zip(input_vec, self.weights))), 0.0) + self.bias)
        
    def train(self, input_vecs, labels, iteration, rate):
        """
        Train the Perceptron.
        :param input_vecs: List of input vectors.
        :param labels: List of true labels.
        :param iteration: Number of training iterations (epochs).
        :param rate: Learning rate.
        """
        for i in range(iteration):
            self._one_iteration(input_vecs, labels, rate)

    def _one_iteration(self, input_vecs, labels, rate):
        """
        Perform one iteration of training (one epoch).
        Updates weights after each sample.
        """
        samples = zip(input_vecs, labels)
        for (input_vec, label) in samples:
            output = self.predict(input_vec)
            self._update_weights(input_vec, output, label, rate)

    def _update_weights(self, input_vec, output, label, rate):
        """
        Update weights and bias based on the prediction error.
        Rule: w_new = w_old + rate * (label - output) * input
              b_new = b_old + rate * (label - output)
        """
        # Calculation Step 1: Calculate error (delta)
        # delta = target_label - predicted_output
        delta = label - output
        
        # Calculation Step 2: Update weights
        # new_weight_i = old_weight_i + learning_rate * delta * input_i
        self.weights = list(map( lambda tp: tp[1] + rate * delta * tp[0], zip(input_vec, self.weights)) )
        
        # Calculation Step 3: Update bias
        # new_bias = old_bias + learning_rate * delta
        self.bias += rate * delta
        #print("label - output = delta:" ,label, output, delta)
        #print("weights ", self.weights)
        #print("bias", self.bias)

def f(x):
    #activate func
    # Step function: returns 1 if x > 0.5, else 0
    return 1 if x > 0.5 else 0

def train_and_perceptron(iteration, rate):
    """
    Train a Perceptron to learn the AND function.
    """
    p = Perceptron(2, f)
    input_vecs, labels = [[1,1], [0,0], [1,0], [0,1]], [1, 0, 0, 0]
    p.train(input_vecs, labels, iteration, rate)
    return p

def train_or_perceptron(iteration, rate):
    """
    Train a Perceptron to learn the OR function.
    """
    p = Perceptron(2, f)
    input_vecs, labels = [[1,1], [0,0], [1,0], [0,1]], [1, 0, 1, 1]
    p.train(input_vecs, labels, iteration, rate)
    return p
	
iteration = 10
learning_rate = 0.25
and_perception = train_and_perceptron(iteration, learning_rate)
or_perception = train_or_perceptron(iteration, learning_rate)

print('Perceptron result:')
print ('1 and 1 = %d' % and_perception.predict([1, 1]))
print ('0 and 0 = %d' % and_perception.predict([0, 0]))
print ('1 and 0 = %d' % and_perception.predict([1, 0]))
print ('0 and 1 = %d' % and_perception.predict([0, 1]))
print ('1 or 1 = %d' % or_perception.predict([1, 1]))
print ('0 or 0 = %d' % or_perception.predict([0, 0]))
print ('1 or 0 = %d' % or_perception.predict([1, 0]))
print ('0 or 1 = %d' % or_perception.predict([0, 1]))


import numpy as np

class AdlineGD(object):
    """AdlineGD Classifier (Adaptive Linear Neuron with Gradient Descent)"""
    def __init__(self,eta=0.25,n_iter=10):
        """
        Initialize AdalineGD.
        :param eta: Learning rate (between 0.0 and 1.0).
        :param n_iter: Passes over the training dataset.
        """
        self.eta = eta
        self.n_iter = n_iter

    def fit(self,X,y):
        """
        Fit training data.
        :param X: Training vectors, shape = [n_samples, n_features].
        :param y: Target values, shape = [n_samples].
        :return: self
        """
        self.w_ = np.zeros(X.shape[1])
        self.b_ = np.zeros(1)
        self.cost_ = []

        for _ in range(self.n_iter):
            # Process Flow:
            # 1. Net Input -> 2. Error Calculation -> 3. Cost Calculation -> 4. Weight Update
            
            # 1. Calculate net input: Z = W^T * X + b
            output = self.net_input(X)
            
            # 2. Calculate errors: Error = y - Z
            errors = y - output
            
            # 3. Calculate cost (Sum of Squared Errors / 2): J(w) = 1/2 * sum((y - Z)^2)
            cost = (errors**2).sum()/2.0
            self.cost_.append(cost)
            
            # 4. Update weights based on the gradient of the cost function
            # Gradient = -sum(error * x)
            # w = w - eta * (-sum(error * x)) = w + eta * X.T.dot(errors)
            self.w_ += self.eta * np.dot(X.T,errors)
            
            # Update bias: b = b + eta * sum(errors)
            self.b_ += self.eta * errors.sum()
        return self

    def net_input(self,x):
        """Calculate net input (dot product of inputs and weights plus bias)"""
        return np.dot(x,self.w_) + self.b_

    def activation(self,x):
        """Compute linear activation (identity function for Adaline)"""
        return self.net_input(x)

    def predict(self,x):
        """Return class label after unit step"""
        return np.where(self.activation(x) > 0.5, 1, 0)

# Train AdalineGD for OR function
or_adline = AdlineGD()
or_adline.fit(np.array([[1,1], [0,0], [1,0], [0,1]]), [1, 0, 1, 1])

# Train AdalineGD for AND function
and_adline = AdlineGD()
and_adline.fit(np.array([[1,1], [0,0], [1,0], [0,1]]), [1, 0, 0, 0])

print ('ADLINE result:')
print ('1 and 1 = %d' % and_adline.predict([1, 1]).item())
print ('0 and 0 = %d' % and_adline.predict([0, 0]).item())
print ('1 and 0 = %d' % and_adline.predict([1, 0]).item())
print ('0 and 1 = %d' % and_adline.predict([0, 1]).item())
print ('1 or 1 = %d' % or_adline.predict([1, 1]).item())
print ('0 or 0 = %d' % or_adline.predict([0, 0]).item())
print ('1 or 0 = %d' % or_adline.predict([1, 0]).item())
print ('0 or 1 = %d' % or_adline.predict([0, 1]).item())

Perceptron result:
1 and 1 = 1
0 and 0 = 0
1 and 0 = 0
0 and 1 = 0
1 or 1 = 1
0 or 0 = 0
1 or 0 = 1
0 or 1 = 1
ADLINE result:
1 and 1 = 1
0 and 0 = 0
1 and 0 = 0
0 and 1 = 0
1 or 1 = 1
0 or 0 = 0
1 or 0 = 1
0 or 1 = 1
